In [13]:
import pandas as pd
import numpy as np

## **Hatt+ 2023**

### Data Description

```
   Bytes Format Units   Label     Explanations

   1-  9  I9    ---     TIC       TESS Input Catalogue Identifier
  11- 12  I2    ---     NSectors  [1/26] Number of sectors of TESS data used
      16  I1    ---     RP        [0/1] Repeating pattern module flag
      20  I1    ---     PE        [0/1] Power excess module flag
  24- 30  F7.2  uHz     numax     [3.72/3344.47]? Frequency at maximum power
  32- 37  F6.2  uHz   e_numax     ? Uncertainty on frequency at maximum power
  39- 44  F6.2  uHz     dnu       [0.61/141.13]? Large frequency separation
  46- 49  F4.2  uHz   e_dnu       ? Uncertainty on large frequency separation
  51- 57  A7    ---     sample    Sample target belongs to (20-sec or 120-sec)
  59- 61  A3    ---     Flag      Cross check results (1)
```

### CDS X-Matching

Read-in xmatch.

In [14]:
array_full = pd.read_csv("../0_inputdata_CDSxmatch/Hatt2023_cdsxmatch.csv",low_memory=False)

Remove flagged entries.

In [15]:
array_full = array_full[
    (array_full["RP"] == 1) &
    (array_full["PE"] == 1) &
    (array_full["Flag"].isna())
]
print(f"Cleaned data length: {len(array_full):.0f}")

Cleaned data length: 2181


Curate simplfied array.

In [16]:
h23 = pd.DataFrame({
    # star identifiers
    "gaiadr3_id": array_full["Source"],
    "ra": array_full["RAdeg"],
    "dec": array_full["DEdeg"],
    # asteroseismic parameters
    "numax": array_full["numax"],
    "e_numax": array_full["e_numax"],
    "dnu": array_full["dnu"],
    "e_dnu": array_full["e_dnu"],
    "ev_state": np.full(len(array_full), np.nan), # unknown ev state
    # Gaia parameters
    "GAIA-teff": array_full["Teff"],
    "GAIA-e_teff": (array_full["B_Teff"] - array_full["b_Teff"]) / 2,
    "GAIA-logg": array_full["logg"],
    "GAIA-e_logg": (array_full["B_logg"] - array_full["b_logg"]) / 2,
    "GAIA-fe_h": array_full["[Fe/H]"],
    "GAIA-e_fe_h": (array_full["B_[Fe/H]"] - array_full["b_[Fe/H]"]) / 2,
    # datasets
    "h23": np.full(len(array_full), int(1)),
    "s26": np.full(len(array_full), int(0)),
    "w26": np.full(len(array_full), int(0)),
    "y18": np.full(len(array_full), int(0)),
    "z22": np.full(len(array_full), int(0)),
    # GSP Spec parameters
    "GSPSPEC-teff": array_full["Teff-S"],
    "GSPSPEC-e_teff": (array_full["B_Teff-S"] - array_full["b_Teff-S"]) / 2,
    "GSPSPEC-logg": array_full["logg-S"],
    "GSPSPEC-e_logg": (array_full["B_logg-S"] - array_full["b_logg-S"]) / 2,
    "GSPSPEC-fe_h": array_full["[Fe/H]-S"],
    "GSPSPEC-e_fe_h": (array_full["B_[Fe/H]-S"] - array_full["b_[Fe/H]-S"]) / 2,
    "GSPSPEC-si_fe": array_full["[Si/Fe]-S"],
    "GSPSPEC-e_si_fe": (array_full["B_[Si/Fe]-S"] - array_full["b_[Si/Fe]-S"]) / 2,
    "GSPSPEC-ca_fe": array_full["[Ca/Fe]-S"],
    "GSPSPEC-e_ca_fe": (array_full["B_[Ca/Fe]-S"] - array_full["b_[Ca/Fe]-S"]) / 2,
    "GSPSPEC-ti_fe": array_full["[Ti/Fe]-S"],
    "GSPSPEC-e_ti_fe": (array_full["B_[Ti/Fe]-S"] - array_full["b_[Ti/Fe]-S"]) / 2,
    "GSPSPEC-mg_fe": array_full["[Mg/Fe]-S"],
    "GSPSPEC-e_mg_fe": (array_full["B_[Mg/Fe]-S"] - array_full["b_[Mg/Fe]-S"]) / 2,
    "GSPSPEC-nd_fe": array_full["[Nd/Fe]-S"],
    "GSPSPEC-e_nd_fe": (array_full["B_[Nd/Fe]-S"] - array_full["b_[Nd/Fe]-S"]) / 2,
    "GSPSPEC-s_fe": array_full["[S/Fe]-S"],
    "GSPSPEC-e_s_fe": (array_full["B_[S/Fe]-S"] - array_full["b_[S/Fe]-S"]) / 2,
    "GSPSPEC-zr_fe": array_full["[Zr/Fe]-S"],
    "GSPSPEC-e_zr_fe": (array_full["B_[Zr/Fe]-S"] - array_full["b_[Zr/Fe]-S"]) / 2,
    "GSPSPEC-n_fe": array_full["[N/Fe]-S"],
    "GSPSPEC-e_n_fe": (array_full["B_[N/Fe]-S"] - array_full["b_[N/Fe]-S"]) / 2,
    "GSPSPEC-cr_fe": array_full["[Cr/Fe]-S"],
    "GSPSPEC-e_cr_fe": (array_full["B_[Cr/Fe]-S"] - array_full["b_[Cr/Fe]-S"]) / 2,
    "GSPSPEC-ce_fe": array_full["[Ce/Fe]-S"],
    "GSPSPEC-e_ce_fe": (array_full["B_[Ce/Fe]-S"] - array_full["b_[Ce/Fe]-S"]) / 2,
    "GSPSPEC-ni_fe": array_full["[Ni/Fe]-S"],
    "GSPSPEC-e_ni_fe": (array_full["B_[Ni/Fe]-S"] - array_full["b_[Ni/Fe]-S"]) / 2,
})

h23 = h23.dropna(subset=["e_numax", "e_dnu"])

Taken within $0.75\times$ and $1.35\times$ of $\Delta\nu$-$\nu_\mathrm{max}$ relation (Stello+ 09, Mosser+ 11):

$$ \Delta\nu = 0.27 \nu_\mathrm{max}^{0.76}. $$

In [17]:
def relation(numax):
    coeff = 0.27
    return coeff * numax**(0.76)

mask = (h23["dnu"] < 1.35*relation(h23["numax"])) & (h23["dnu"] > 0.75*relation(h23["numax"]))

h23 = h23[mask]

Write to `.csv` file.

In [18]:
h23.to_csv("./hatt23_asteroseismic.csv", index=False)